# S5+classification

학습이 끝난 S5 분할기는 고정한다. μ에 헤드를 달아 슬라이스에 병변이 있을 확률을 내고, 그 확률이 검증에서 고른 τ보다 낮으면 마스크를 비운다.

1. 고정된 S5에서 μ를 꺼낸다.
2. `LesionPresenceHead`가 로짓을 낸다. 시그모이드는 헤드 안에 없다.
3. 손실은 `BCEWithLogits`이다. 양성 가중치는 2이다.
4. 확률은 `p = sigmoid(로짓)`이다.
5. 검증 Dice가 최대인 τ로 테스트 마스크를 게이트한다.

S5 가중치는 `results/covid/label_ratio/rXX/s5/seg_best.pt`에 있어야 한다. 이 노트북은 그 파일을 다시 학습하지 않는다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import torch
import torch.nn as nn

from pipeline.config import RESULT_DIR
from pipeline.engine import DEVICE, atomic_torch_save, batch_dice, batch_pred_gt_sums, safe_load
from pipeline.metrics import binary_auroc, metrics_bundle
from pipeline.models_skip import VAE as SkipVAE
from pipeline.stage_segment import make_bundle

RATIO = 0.3
CLS_EPOCHS = 40
CLS_ES = 10
CLS_POS_WEIGHT = 2.0
RUN = False  # True로 바꾸면 헤드를 학습한다. report.json이 있으면 건너뛴다.

## 헤드

μ는 `1024 = 32×32` 채널의 공간 특징이다. 평균 풀링과 최대 풀링을 이어 붙여 로짓 하나를 낸다.

In [ ]:
class LesionPresenceHead(nn.Module):
    def __init__(self, in_channels: int, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(in_channels * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, feature: torch.Tensor) -> torch.Tensor:
        pooled = torch.cat([
            self.avg_pool(feature).flatten(1),
            self.max_pool(feature).flatten(1),
        ], dim=1)
        return self.classifier(pooled).squeeze(1)

## 확률과 τ

마스크 로짓은 S5 디코더가 만든다. 존재 확률은 같은 μ에서 따로 만든다. τ는 검증 집합에서 전체 평균 Dice가 가장 높은 값이다.

In [ ]:
def prep_xy(batch):
    x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
    if x.ndim == 3:
        x = x.unsqueeze(1)
    if y.ndim == 3:
        y = y.unsqueeze(1)
    return x, y


@torch.inference_mode()
def collect(model, head, loader):
    model.eval()
    head.eval()
    bags = {"gt": [], "mass": [], "dice": [], "score": []}
    for batch in loader:
        x, y = prep_xy(batch)
        mu, _lv, _z, skips = model.encoder.forward_features(x)
        seg_logits = model.decoder(mu, skips)
        mass, gt = batch_pred_gt_sums(seg_logits, y)
        bags["gt"].append(gt)
        bags["mass"].append(mass)
        bags["dice"].append(batch_dice(seg_logits, y))
        bags["score"].append(torch.sigmoid(head(mu)).detach().cpu().numpy())
    return {k: np.concatenate(v) for k, v in bags.items()}


def pick_tau(val):
    best = None
    for i in range(1, 100):
        thr = i / 100.0
        keep = val["score"] >= thr
        m = metrics_bundle(val["gt"], val["mass"], val["dice"], keep)
        key = (m["overall_dice"], -m["empty_fp_rate"])
        if best is None or key > best["key"]:
            best = {"thr": thr, "key": key, "m": m}
    return best["thr"], best["m"]

## 학습

타깃은 슬라이스에 병변 픽셀이 있으면 1, 없으면 0이다. 손실에 넣는 값은 확률이 아니라 로짓이다. 검증 AUROC가 10 에폭 동안 오르지 않으면 멈춘다.

In [ ]:
def train_s5_cls(ratio=RATIO):
    pct = int(round(ratio * 100))
    seg_ckpt = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5" / "seg_best.pt"
    out_dir = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5_cls"
    out_dir.mkdir(parents=True, exist_ok=True)
    report_path = out_dir / "report.json"
    if report_path.exists():
        print("skip", report_path)
        return json.loads(report_path.read_text(encoding="utf-8"))

    _full, train_loader, val_loader, test_loader, n_train = make_bundle(ratio, 4)
    model = SkipVAE(base=32).to(DEVICE)
    model.load_state_dict(safe_load(seg_ckpt, map_location="cpu")["model"])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    head = LesionPresenceHead(32 * 32).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CLS_POS_WEIGHT], device=DEVICE))
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auroc, bad, best_ep = -1.0, 0, 0
    best_path = out_dir / "head_best.pt"
    trend = []

    for ep in range(1, CLS_EPOCHS + 1):
        head.train()
        total, n = 0.0, 0
        for batch in train_loader:
            x, y = prep_xy(batch)
            with torch.no_grad():
                mu, _lv, _z, _sk = model.encoder.forward_features(x)
            y_cls = (y.flatten(1).sum(1) > 0).float()
            opt.zero_grad(set_to_none=True)
            loss = bce(head(mu), y_cls)
            loss.backward()
            opt.step()
            total += float(loss.item()) * y_cls.size(0)
            n += int(y_cls.size(0))
        val = collect(model, head, val_loader)
        auroc = binary_auroc(val["gt"] > 0, val["score"])
        trend.append({"ep": ep, "train_bce": total / max(n, 1), "val_auroc": auroc})
        print(f"ep {ep} bce={trend[-1]['train_bce']:.4f} auroc={auroc:.4f}")
        if auroc > best_auroc + 1e-4:
            best_auroc, bad, best_ep = auroc, 0, ep
            atomic_torch_save({"head": head.state_dict(), "ep": ep, "auroc": auroc}, best_path)
        else:
            bad += 1
            if bad >= CLS_ES:
                break

    head.load_state_dict(safe_load(best_path, map_location="cpu")["head"])
    val = collect(model, head, val_loader)
    test = collect(model, head, test_loader)
    tau, _val_m = pick_tau(val)
    mt = metrics_bundle(test["gt"], test["mass"], test["dice"], test["score"] >= tau)
    report = {
        "ratio": ratio,
        "n_train": n_train,
        "loss": "BCEWithLogits",
        "probability": "p = sigmoid(Head(mu)) = P(lesion present | slice)",
        "tau": tau,
        "best_ep": best_ep,
        "val_auroc": best_auroc,
        "test_dice": float(mt["overall_dice"]),
        "test_nonempty_dice": float(mt["nonempty_dice"]),
        "test_empty_fp_rate": float(mt["empty_fp_rate"]),
    }
    (out_dir / "prob_trend.json").write_text(json.dumps(trend, indent=2), encoding="utf-8")
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(report)
    return report


if RUN:
    train_s5_cls(RATIO)